In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scanpy as sc
import leidenalg
import scipy.sparse as sp
import celltypist

In [ ]:
adata_all = sc.read_h5ad('adatas_final.h5ad')

In [ ]:
#celltypist annotation
temp_adata = sc.AnnData(X=adata_all.layers['filtered'].copy(),
                        obs=adata_all.obs,
                        var=adata_all.var)


sc.pp.normalize_total(temp_adata, target_sum = 1e4)
sc.pp.log1p(temp_adata)

model = celltypist.models.Model.load("Immune_All_Low.pkl")
prediction = celltypist.annotate(temp_adata, model = model, majority_voting = True)

adata_all.obs['cell_type_label'] = prediction.predicted_labels.loc[adata_all.obs.index, 'majority_voting']
adata_all.obs['cell_type_conf_score'] = prediction.probability_matrix.loc[adata_all.obs.index].max(axis=1)

del temp_adata

In [ ]:
sc.pl.umap(adata_all, color=['cell_type_label', 'cell_type_conf_score'], frameon=False, wspace=0.2)

In [ ]:
sc.pp.normalize_total(adata_all)
sc.pp.log1p(adata_all)
sc.pp.neighbors(adata_all, use_rep='X_scanvi')
sc.tl.leiden(adata_all, resolution=.5, key_added='res_05', flavor='igraph', n_iterations=2)
sc.tl.leiden(adata_all, resolution=.75, key_added='res_075', flavor='igraph', n_iterations=2)
sc.tl.leiden(adata_all, resolution=1, key_added='res_1', flavor='igraph', n_iterations=2)
sc.tl.leiden(adata_all, resolution=1.25, key_added='res_125', flavor='igraph', n_iterations=2)
sc.tl.umap(adata_all)
sc.pl.umap(adata_all, color=['res_05','res_075','res_1', 'res_125'], frameon=False, wspace=0.2,  legend_loc= 'on data')

In [ ]:
sc.tl.rank_genes_groups(adata_all, groupby='res_125', method='wilcoxon', key_added='dea_res_125')
sc.pl.rank_genes_groups_dotplot(adata_all,key='dea_res_125', groupby='res_125', standard_scale='var', n_genes=6)

In [ ]:
#cluster 15 was red blood cell contamination and cluster 19 were platelets so i removed them
clusters_to_keep = [c for c in adata_all.obs['res_125'].unique() if c not in ['15', '19']]
adata_all = adata_all[adata_all.obs['res_125'].isin(clusters_to_keep)].copy()

In [ ]:
#reprocessing my umap becaue i filtered out clusters 15 & 19
sc.pp.neighbors(adata_all, use_rep='X_scanvi')
sc.tl.leiden(adata_all, resolution=1.25, key_added='res_1_25', flavor='igraph', n_iterations=2)
sc.tl.umap(adata_all)
sc.pl.umap(adata_all, color=['res_1_25'], frameon=False, legend_loc= 'on data')

In [ ]:
sc.tl.rank_genes_groups(adata_all, groupby='res_1_25', method='wilcoxon', key_added='dea_res_1_25')
sc.pl.rank_genes_groups_dotplot(adata_all,key='dea_res_1_25', groupby='res_1_25', standard_scale='var', n_genes=6)

In [ ]:
#printing out a list of the top markers
#many of the covid cells are highly transcriptional and therefore have
#rna genes as some of their top markers, i masked these
#along with any possible mt genes to only get the unique cell markers
markers_df = pd.DataFrame(adata_all.uns['dea_res_125']['names'])
is_housekeeping = markers_df.map(lambda x: str(x).startswith(('RNA18S5', 'RNA28S5','RPL', 'RPS', 'MT-'))) #masks ribosomal genes
clean_markers_df = markers_df[~is_housekeeping].apply(lambda x: pd.Series(x.dropna().values))

print("--- Top Marker Genes for Clusters ---")
print(clean_markers_df.head(10).to_string())

In [ ]:
manual_annotation ={"0": "Cytotoxic CD8+ T Cells",
                    "1": "Naive CD4+ T Cells",
                    "2": "Resting Naive CD4+ T Cells",
                    "3": "Natural Killer Cells",
                    "4": "Classical Monocytes",
                    "5": "Conventional Dendritic Cells",
                    "6": "Primed Naive CD4+ T Cells",
                    "7": "Activated Inflammatory Classical Monocytes",
                    "8": "Non-Classical Monocytes",
                    "9": "Antibody-Secreting Plasma Cells",
                    "10": "ER-Stressed Plasma Cells",
                    "11": "Mature B Cells",
                    "12": "Natural Killer Cells",
                    "13": "Cycling Cells",
                    "14": "Extravasating(Migratory) Classical Monocytes",
                    "15": "Mobilized Hematopoietic Progenitors",
                    "16": "Plasmacytoid Dendritic Cells",
                    "17": "Low-Density Granulocytes/Neutrophils"

}

In [ ]:
#mapped my manual annotations to the clusters on the umap
adata_all.obs['manual_cell_type'] = adata_all.obs['res_125'].map(manual_annotation)
sc.pl.umap(adata_all, color = ['manual_cell_type'], frameon = False, title = 'Covid Vs Healthy PBMC Cell Type CLusters')